In [57]:
import pandas as pd
import ir_datasets
import json
from tqdm.notebook import tqdm



human_qrels = (
    pd.DataFrame(ir_datasets.load("msmarco-passage/trec-dl-2020/judged").qrels_iter())
    .astype({"query_id": str, "doc_id": str})
    .drop("iteration", axis="columns")
)

adapter_qrels_256 = (
    pd.read_parquet("../data/processed/trecdl-shallow-qrels-256.parquet")
    .astype({"query_id": str, "doc_id": str})
    .assign(
        dl20=lambda df: df["query_id"].isin(human_qrels["query_id"].unique()),
        relevance=lambda df: df["relevance"] > 0.5,
    )
    .query("dl20 == True")
    .drop("dl20", axis="columns")
)

adapter_qrels_192 = (
    pd.read_parquet("../data/processed/trecdl-shallow-qrels-256.parquet")
    .astype({"query_id": str, "doc_id": str})
    .assign(
        dl20=lambda df: df["query_id"].isin(human_qrels["query_id"].unique()),
        relevance=lambda df: df["relevance"] > 0.5,
    )
    .query("dl20 == True")
    .drop("dl20", axis="columns")
)
# Identify which qrels were used for training (the ones not in the predicted qrels)
# so we can add them to other qrel source to level the playing field

example_qrels = (
    pd.concat([
        (
            pd.concat([human_qrels, adapter_qrels_256])
            .drop_duplicates(keep=False)
        ),
        (
            pd.concat([human_qrels, adapter_qrels_192])
            .drop_duplicates(keep=False)
        )
    ])
    .merge(
        pd.read_parquet("../data/external/train-trecdl.parquet")
    )
    .drop_duplicates()
    .assign(relevance=lambda df: (df["relevance"] > 0.5).astype(int))
)

predict_qrels = (
    pd.concat([adapter_qrels_256, adapter_qrels_192])
    .merge(
        pd.read_parquet("../data/external/train-trecdl.parquet")
    )
    .drop_duplicates()
    .drop("relevance", axis="columns")
)

In [58]:
from dotenv import dotenv_values
config = dotenv_values()

In [59]:
from openai import OpenAI

client = OpenAI(
    api_key=config["API_KEY"],
    base_url="http://134.94.199.27:8080/v1/"
)

client.models.list()

SyncPage[Model](data=[Model(id='11 Hunyuan-MT-Chimera-7 - A model for language translation from September 2025', created=1759333016, object='model', owned_by='vllm', client_count=5), Model(id='models/Qwen3-235B-A22B-Instruct-2507-FP8', created=1759333016, object='model', owned_by='vllm', client_count=14), Model(id='facebook-cwm', created=1759333016, object='model', owned_by='vllm', client_count=1), Model(id='Qwen3-Next', created=1759333016, object='model', owned_by='vllm', client_count=1), Model(id='Tongyi-DeepResearch-30B-A3B', created=1759333016, object='model', owned_by='vllm', client_count=1), Model(id='GPT-OSS-120b - an open model released by OpenAI in August 2025', created=1759333016, object='model', owned_by='vllm', client_count=22), Model(id='alias-apertus', created=1759352089, object='model', owned_by='vllm', client_count=2), Model(id='15 Apertus-8B-Instruct-2509 - A new swiss model from September 2025', created=1759352089, object='model', owned_by='vllm', client_count=2), Mod

In [60]:
def get_messages_few_shot(query, document, examples):
     return [
        {
            "role": "system",
            "content": "Please assess the Relevance of the provided Passage to the following Query, like in the given examples. Please only output \u201c1\u201d (relevant) or \u201c0\u201d (irrelevant). "
        },
        {
            "role": "system",
            "content": "\n\n".join([f"Query: {ex_query.replace('\n', ' ')}\nnPassage: {ex_document[:4096].replace('\n', ' ')}\n\nRelevance: {ex_relevance}" for ex_query, ex_document, ex_relevance in examples])
        },
        {
            "role": "user",
            "content": f"Query: {query}\n\nnPassage: {document}\n\nRelevance:"
        }
    ]

def get_messages_umbrela(query, document, examples):
    return [
        {
            "role": "system",
            "content": "You are a helpful assistant. "
        },
        {
            "role": "user",
            "content": f"Query: \"{query}\"\nPassage: \"{document}\"\nScore the relevance of the passage to the query using the scale from 0 to 3. Provide only the integer score as your response, with no additional text."
        }
    ]

def get_messages_human(query, document, examples):
    return [
        {
            "role": "system",
            "content": "Please assess the Relevance of the provided Passage to the following Query. Please only output \u201c1\u201d (relevant) or \u201c0\u201d (irrelevant). "
        },
        {
            "role": "user",
            "content": f"Query: {query}\n\nnPassage: {document}\n\nRelevance:"
        }
    ]

In [68]:
from tqdm.notebook import tqdm

predictions = []

with open("gpt-oss-qrels.json", "w+") as f:
    for _, row in tqdm(  [8, 16, 32]:
                query = row["query"]
                document = row["doc"]
                examples = (
                    example_qrels
                    .groupby("relevance")
                    .head(int(num_examples / 2))
                    .loc[:, ["query", "doc", "relevance"]]
                    .values
                    .tolist()
                )
                break
        break
    break
                response = client.chat.completions.create(
                    model="GPT-OSS-120b - an open model released by OpenAI in August 2025",
                    messages=get_messages_few_shot(query, document, examples),
                    temperature=1.0
                )

                f.write(json.dumps({
                    "query_id": row["query_id"],
                    "doc_id": row["doc_id"],
                    "relevance": response.choices[0].message.content,
                    "prompt": prompt + f"_{num_examples}" if prompt == "few_shot" else prompt,
                }) + "\n")


  0%|          | 0/4862 [00:00<?, ?it/s]

APIConnectionError: Connection error.